In [ ]:
"""
Sarcasm-Aware Sentiment Analysis Mini-Project
==============================================
Demonstrates why standard sentiment analysis fails on sarcasm,
and implements a contrast-aware classifier as a mitigation.

Requirements: numpy (pip install numpy)
Optional: scikit-learn for train/test split (pip install scikit-learn)
"""

import re
import numpy as np
from typing import List, Tuple, Dict

# ============================================================
# PART 1: BASELINE LEXICON SENTIMENT ANALYZER
# ============================================================

# Minimal sentiment lexicon (in practice, use VADER or AFINN)
POSITIVE_WORDS = {
    "love", "great", "amazing", "wonderful", "fantastic", "perfect",
    "awesome", "brilliant", "delightful", "thrilled", "adore", "best",
    "good", "nice", "excellent", "happy", "glad", "enjoy", "beautiful",
    "saved", "better", "improved", "success", "win", "congratulations"
}

NEGATIVE_WORDS = {
    "hate", "terrible", "awful", "worst", "boring", "annoying",
    "miserable", "dreadful", "horrible", "ugh", "bad", "poor",
    "disappointing", "failed", "disaster", "angry", "sad", "upset",
    "broken", "useless", "waste", "stupid", "ridiculous"
}

def tokenize(text: str) -> List[str]:
    """Simple whitespace + punctuation tokenizer."""
    text = text.lower()
    text = re.sub(r"[^\w\s']", " ", text)
    return text.split()

def lexicon_sentiment(text: str) -> Dict:
    """
    Baseline: count positive and negative words, return polarity.
    This is the approach used by VADER, TextBlob, and similar systems.
    """
    tokens = tokenize(text)
    pos_count = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg_count = sum(1 for t in tokens if t in NEGATIVE_WORDS)

    if pos_count > neg_count:
        polarity = "positive"
        score = (pos_count - neg_count) / max(len(tokens), 1)
    elif neg_count > pos_count:
        polarity = "negative"
        score = (neg_count - pos_count) / max(len(tokens), 1)
    else:
        polarity = "neutral"
        score = 0.0

    return {
        "polarity": polarity,
        "score": round(score, 3),
        "pos_hits": pos_count,
        "neg_hits": neg_count,
        "tokens": tokens
    }


# ============================================================
# PART 2: CONTRAST-AWARE FEATURE EXTRACTOR
# ============================================================

# Sarcasm often involves positive sentiment applied to an unpleasant situation.
# We define a set of "unpleasant situation" words to capture this contrast.

UNPLEASANT_SITUATIONS = {
    "monday", "mondays", "traffic", "meeting", "meetings", "homework",
    "taxes", "dentist", "queue", "delay", "delayed", "chores", "rain",
    "bill", "bills", "overtime", "exam", "exams", "deadline", "commute",
    "laundry", "dishes", "insomnia", "allergy", "flu", "cold", "broken",
    "cancelled", "cancelled", "stuck", "waiting", "crowded", "smelly",
    "smeared", "paint", "protest", "senate", "politician", "election"
}

# Sarcasm markers: words that often signal ironic intent
SARCASM_MARKERS = {
    "oh", "wow", "yeah", "sure", "totally", "obviously", "clearly",
    "definitely", "absolutely", "fantastic", "great", "perfect",
    "thanks", "brilliant", "genius", "love", "adore"
}

def extract_contrast_features(text: str) -> Dict:
    """
    Extract features that capture the contrast pattern typical of sarcasm:
    positive sentiment word co-occurring with an unpleasant situation.
    """
    tokens = tokenize(text)
    token_set = set(tokens)

    pos_count = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg_count = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    unpleasant_count = sum(1 for t in tokens if t in UNPLEASANT_SITUATIONS)
    marker_count = sum(1 for t in tokens if t in SARCASM_MARKERS)

    # The core contrast signal: positive words + unpleasant situation
    has_contrast = 1.0 if (pos_count > 0 and unpleasant_count > 0) else 0.0

    # A stronger version: positive word immediately followed by unpleasant noun
    # (captures "love Mondays" style patterns)
    strong_contrast = 0.0
    for i in range(len(tokens) - 1):
        if tokens[i] in POSITIVE_WORDS and tokens[i+1] in UNPLEASANT_SITUATIONS:
            strong_contrast = 1.0
            break

    # Exclamation + positive word in a negative context (overstatement)
    exclamation = 1.0 if "!" in text else 0.0

    return {
        "pos_count": pos_count,
        "neg_count": neg_count,
        "unpleasant_count": unpleasant_count,
        "marker_count": marker_count,
        "has_contrast": has_contrast,
        "strong_contrast": strong_contrast,
        "exclamation": exclamation,
        "text_length": len(tokens)
    }


# ============================================================
# PART 3: TRAINING DATA (CURATED EXAMPLES)
# ============================================================

# Label: 1 = sarcastic, 0 = non-sarcastic
# These are illustrative examples. In a real project, use iSarcasm,
# Multi-Sarcasm, or a domain-specific annotated dataset.

TRAINING_DATA = [
    # Sarcastic examples (contrast patterns)
    ("I love waking up at 4 AM for a delayed flight", 1),
    ("Great, another Monday morning meeting", 1),
    ("Oh fantastic, the traffic is wonderful today", 1),
    ("Perfect, my exam got cancelled after I studied for weeks", 1),
    ("Love how the dentist appointment is always on my day off", 1),
    ("Brilliant idea to schedule the deadline during vacation", 1),
    ("Wow, another queue at the grocery store, so exciting", 1),
    ("Amazing, the rain ruined my laundry again", 1),
    ("Thrilled to be stuck in traffic for two hours", 1),
    ("Best day ever, my phone broke and it's raining", 1),
    ("Thank you so much for the tax bill, really appreciate it", 1),
    ("Sure, because another meeting will definitely solve everything", 1),
    ("Delightful, the internet went down during my presentation", 1),
    ("Adore how the bills keep coming even when I'm broke", 1),
    ("Wonderful, the Senate saved the climate again today", 1),
    ("So glad I spent an hour waiting on hold", 1),

    # Non-sarcastic positive examples
    ("I love this restaurant, the food is amazing", 0),
    ("Great job on the presentation, it was excellent", 0),
    ("The weather is beautiful today", 0),
    ("Congratulations on your success, well deserved", 0),
    ("I really enjoyed the movie, it was wonderful", 0),
    ("Best vacation ever, perfect weather and great food", 0),
    ("Thank you for the help, I really appreciate it", 0),
    ("The new phone is amazing, the camera is fantastic", 0),
    ("Thrilled to announce my promotion", 0),
    ("Wonderful news, the project was approved", 0),
    ("Adore this book, the writing is brilliant", 0),
    ("Perfect score on the exam, all that studying paid off", 0),

    # Non-sarcastic negative examples
    ("I hate Monday mornings, they are terrible", 0),
    ("The traffic was awful, I was stuck for hours", 0),
    ("Terrible meeting, completely useless", 0),
    ("The dentist appointment was painful and miserable", 0),
    ("Horrible experience at the restaurant, the food was bad", 0),
    ("My exam was a disaster, I failed", 0),
    ("The rain ruined everything, worst day ever", 0),
    ("Annoying queue, waited forever", 0),
    ("Disappointing movie, boring and predictable", 0),
    ("The internet broke during my presentation, disaster", 0),
    ("Useless phone, the battery is terrible", 0),
    ("Sad news, the project was cancelled", 0),
]


# ============================================================
# PART 4: LOGISTIC REGRESSION CLASSIFIER (FROM SCRATCH)
# ============================================================

class SarcasmClassifier:
    """
    Binary logistic regression for sarcasm detection using
    contrast-aware features. Trained with full-batch gradient descent.
    """

    def __init__(self, learning_rate: float = 0.1, epochs: int = 1000, l2: float = 0.01):
        self.lr = learning_rate
        self.epochs = epochs
        self.l2 = l2
        self.weights = None
        self.bias = 0.0
        self.feature_names = None

    def _featurize(self, text: str) -> np.ndarray:
        """Convert text to feature vector."""
        features = extract_contrast_features(text)
        return np.array([
            features["pos_count"],
            features["neg_count"],
            features["unpleasant_count"],
            features["marker_count"],
            features["has_contrast"],
            features["strong_contrast"],
            features["exclamation"],
            features["text_length"],
        ], dtype=float)

    def _featurize_batch(self, texts: List[str]) -> np.ndarray:
        return np.array([self._featurize(t) for t in texts])

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        # Clip for numerical stability
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, texts: List[str], labels: List[int]):
        """Train the classifier."""
        X = self._featurize_batch(texts)
        y = np.array(labels, dtype=float)
        n, d = X.shape

        # Standardize features for stable gradient descent
        self.mu = X.mean(axis=0)
        self.sd = X.std(axis=0) + 1e-8
        X = (X - self.mu) / self.sd

        self.weights = np.zeros(d)
        self.bias = 0.0

        for epoch in range(self.epochs):
            z = X @ self.weights + self.bias
            p = self._sigmoid(z)

            # Gradients
            grad_w = (X.T @ (p - y)) / n + self.l2 * self.weights
            grad_b = np.mean(p - y)

            # Update
            self.weights -= self.lr * grad_w
            self.bias -= self.lr * grad_b

            # Optional: print loss every 200 epochs
            if (epoch + 1) % 200 == 0:
                loss = -np.mean(y * np.log(p + 1e-8) + (1 - y) * np.log(1 - p + 1e-8))
                loss += 0.5 * self.l2 * np.sum(self.weights ** 2)
                # print(f"Epoch {epoch+1}/{self.epochs}, Loss: {loss:.4f}")

        return self

    def predict_proba(self, text: str) -> float:
        """Return probability of sarcasm."""
        x = self._featurize(text).reshape(1, -1)
        x = (x - self.mu) / self.sd
        z = x @ self.weights + self.bias
        return float(self._sigmoid(z)[0])

    def predict(self, text: str, threshold: float = 0.5) -> int:
        """Return 1 if sarcastic, 0 otherwise."""
        return 1 if self.predict_proba(text) >= threshold else 0


# ============================================================
# PART 5: EVALUATION AND DEMONSTRATION
# ============================================================

def evaluate_sarcasm_awareness():
    """
    Compare lexicon sentiment vs. sarcasm-aware classifier on
    examples where literal sentiment and intended sentiment diverge.
    """
    print("=" * 70)
    print("SARCASTIC EXAMPLES: WHERE LEXICON SENTIMENT FAILS")
    print("=" * 70)

    sarcastic_tests = [
        "I love waking up at 4 AM for a delayed flight",
        "Great, another Monday morning meeting",
        "Oh fantastic, the traffic is wonderful today",
        "Perfect, my exam got cancelled after I studied for weeks",
        "Wow, another queue at the grocery store, so exciting",
        "Amazing, the rain ruined my laundry again",
        "Thrilled to be stuck in traffic for two hours",
        "Best day ever, my phone broke and it's raining",
        "Sure, because another meeting will definitely solve everything",
        "Delightful, the internet went down during my presentation",
        "Good morning from the Senate. Even today we saved the climate!",
        "With the Senate smeared, the climate will certainly be better",
    ]

    print(f"{'Text':<55} {'Lexicon':<12} {'Sarc Prob':<10}")
    print("-" * 77)

    for text in sarcastic_tests:
        lex = lexicon_sentiment(text)
        sarc_prob = classifier.predict_proba(text)
        # Truncate for display
        display_text = text if len(text) < 52 else text[:49] + "..."
        print(f"{display_text:<55} {lex['polarity']:<12} {sarc_prob:.3f}")

    print()
    print("=" * 70)
    print("NON-SARCASTIC EXAMPLES: BASELINE BEHAVIOR")
    print("=" * 70)

    sincere_tests = [
        "I love this restaurant, the food is amazing",
        "Great job on the presentation, it was excellent",
        "The weather is beautiful today",
        "I hate Monday mornings, they are terrible",
        "The traffic was awful, I was stuck for hours",
        "Congratulations on your success, well deserved",
    ]

    print(f"{'Text':<55} {'Lexicon':<12} {'Sarc Prob':<10}")
    print("-" * 77)

    for text in sincere_tests:
        lex = lexicon_sentiment(text)
        sarc_prob = classifier.predict_proba(text)
        display_text = text if len(text) < 52 else text[:49] + "..."
        print(f"{display_text:<55} {lex['polarity']:<12} {sarc_prob:.3f}")


def demonstrate_failure_case():
    """
    Show a specific case where the contrast heuristic also fails:
    sarcasm that does not rely on the positive + unpleasant pattern.
    """
    print()
    print("=" * 70)
    print("LIMITATIONS: WHEN CONTRAST HEURISTICS ALSO FAIL")
    print("=" * 70)

    # This tweet uses irony without a positive sentiment word.
    # "I hope so" is not in our positive lexicon, and the sarcasm
    # depends on the double meaning of "last generation."
    hard_examples = [
        ("ultima generazione lo spero proprio",
         "last generation I really hope so",
         "Irony via ambiguity: 'last generation' means both the activist group and literal extinction."),
        ("In realta sono degli artisti lasciate che sfoghino liberamente la loro creativita",
         "In fact they are artists let them freely vent their creativity",
         "Ironic praise: calling vandals 'artists' is a violation of the Maxim of Quality."),
    ]

    for original, translation, explanation in hard_examples:
        lex = lexicon_sentiment(original)
        sarc_prob = classifier.predict_proba(original)
        print(f"\nOriginal: {original}")
        print(f"Translation: {translation}")
        print(f"Lexicon polarity: {lex['polarity']} (score: {lex['score']})")
        print(f"Sarcasm probability: {sarc_prob:.3f}")
        print(f"Why both fail: {explanation}")


def practical_pipeline_demo():
    """
    Demonstrate a practical pipeline: flag low-confidence or high-sarcasm
    predictions for human review rather than forcing a polarity.
    """
    print()
    print("=" * 70)
    print("PRACTICAL PIPELINE: ROUTING AMBIGUOUS CASES TO HUMAN REVIEW")
    print("=" * 70)

    reviews = [
        "The product works exactly as described, very satisfied",
        "Great, the package arrived three weeks late and damaged",
        "Customer service was helpful and resolved my issue quickly",
        "Love how the app crashes every time I try to save",
        "The quality is excellent for the price",
        "Oh wonderful, another update that broke everything",
    ]

    print(f"{'Review':<50} {'Lexicon':<10} {'Sarc':<6} {'Action':<20}")
    print("-" * 86)

    for review in reviews:
        lex = lexicon_sentiment(review)
        sarc_prob = classifier.predict_proba(review)

        # Decision logic: if sarcasm probability is high, do not trust lexicon
        if sarc_prob > 0.7:
            action = "HUMAN REVIEW"
            final = "ambiguous (likely sarcastic)"
        elif sarc_prob > 0.4:
            action = "FLAG FOR REVIEW"
            final = f"tentative: {lex['polarity']}"
        else:
            action = "AUTO-ACCEPT"
            final = lex['polarity']

        display = review if len(review) < 47 else review[:44] + "..."
        print(f"{display:<50} {lex['polarity']:<10} {sarc_prob:<6.2f} {action:<20}")


# ============================================================
# PART 6: TRAIN AND RUN
# ============================================================

if __name__ == "__main__":
    # Prepare training data
    texts = [item[0] for item in TRAINING_DATA]
    labels = [item[1] for item in TRAINING_DATA]

    # Train the sarcasm classifier
    print("Training sarcasm classifier on", len(texts), "examples...")
    classifier = SarcasmClassifier(learning_rate=0.5, epochs=2000, l2=0.01)
    classifier.fit(texts, labels)
    print("Training complete.\n")

    # Run demonstrations
    evaluate_sarcasm_awareness()
    demonstrate_failure_case()
    practical_pipeline_demo()

    # Final note
    print()
    print("=" * 70)
    print("KEY TAKEAWAY")
    print("=" * 70)
    print("""
The lexicon baseline fails on sarcastic text because it treats words in isolation.
The contrast-aware classifier improves detection of 'positive word + unpleasant
situation' patterns but still fails on context-dependent irony, cultural
references, and understatement.

In production, sentiment analysis on social media should:
  1. Use sarcasm-aware benchmarks (iSarcasm, Multi-Sarcasm, BESSTIE) for evaluation
  2. Flag ambiguous predictions for human review
  3. Never report sentiment aggregates without acknowledging the sarcasm
     misclassification rate, which can systematically bias results
  4. Consider conversational context (parent comments, thread history)
     when available, as sarcasm in dialogue often depends on the preceding turn
""")

Training sarcasm classifier on 40 examples...
Training complete.

SARCASTIC EXAMPLES: WHERE LEXICON SENTIMENT FAILS
Text                                                    Lexicon      Sarc Prob 
-----------------------------------------------------------------------------
I love waking up at 4 AM for a delayed flight           positive     0.945
Great, another Monday morning meeting                   positive     0.905
Oh fantastic, the traffic is wonderful today            positive     0.749
Perfect, my exam got cancelled after I studied fo...    positive     0.977
Wow, another queue at the grocery store, so excit...    neutral      0.727
Amazing, the rain ruined my laundry again               positive     0.943
Thrilled to be stuck in traffic for two hours           positive     0.968
Best day ever, my phone broke and it's raining          positive     0.290
Sure, because another meeting will definitely sol...    neutral      0.675
Delightful, the internet went down during my pres..